# Monitoring ice velocity on Vega Island, Antarctica, using Sentinel-1 offset tracking

## Author: Moritz Rath

Geoprocessing with Python, Wintersemester 25/26, Geographisches Institut, Humboldt-Universität zu Berlin
Subervisors: Dr. Dirk Pflugmacher, Eduardo Ribeiro Lacerda

Hand-in due: March 15, 2026

**1. Import Packages**

In [8]:
import os
import ee
import geemap
import datetime
import json
import geopandas as gpd


try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

path = 'C:/Users/morit/Documents/Uni/HU/Python/geopy_final/data'

Import region shapefile for Vega Island

In [13]:
regionSHP = gpd.read_file(os.path.join(path, 'geodata/vega_island_outline.shp'))
regionSHP = regionSHP.to_crs(epsg=4326)
regionSHP = regionSHP.drop(columns=['sourcedate', 'revdate'])
regionJS = json.loads(regionSHP.to_json())
region = ee.FeatureCollection(regionJS)

In [14]:
Map = geemap.Map()
Map.centerObject(region, 10)
Map.addLayer(region, {}, 'Region')
Map

Map(center=[-63.848051711623604, -57.38302088676081], controls=(WidgetControl(options=['position', 'transparen…

Get data from GEE

In [22]:
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(region)
      .filterDate('2024-01-01', '2025-01-01')
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'HH'))
      .select('HH'))

s1

Define quaters and finetuning

In [ ]:
quaters = []

for year in range(2019, 2025):
    for (sm, em) in [(1,3), (4,6), (7,9),(10,12)]:
        start = datetime.date(year, sm, 1)

        if em == 12:
            end = datetime.date(year + 1, 1, 1) - datetime.timedelta(days=1)
        else:
            end = datetime.date(year, 1, 1) - datetime.timedelta(days=1)

        quaters.append((start, end))
